# CHEAT SHEET

### Pandas Dataframe groupby() Method
Groupby là một quy trình xử lý dữ liệu chuẩn hóa, bao gồm 3 bước chính (còn gọi là phương pháp Split-Apply-Combine):
- Split (Chia tách): Phân tách tập dữ liệu lớn thành các nhóm nhỏ hơn dựa trên một tiêu chí cụ thể (ví dụ: chia theo các nhãn trong một cột).
- Apply (Áp dụng): Thực thi một hàm tính toán (ví dụ: hàm tính tổng .sum(), hoặc trung bình .mean()) lên từng nhóm dữ liệu một cách độc lập.
- Combine (Kết hợp): Gộp các kết quả vừa tính toán được từ các nhóm lại thành một cấu trúc dữ liệu mới, tóm tắt và trực quan hơn.

In [ ]:
import pandas as pd 
import numpy as np
# 1. Khởi tạo dữ liệu 
data = {
    "Patient_ID": ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08"],
    "MGMT_Status": ["Methylated", "Unmethylated", "Methylated", "Unmethylated", "Methylated", "Unmethylated", "Methylated", "Unmethylated"],
    "Tumor_Grade": ["Grade 2", "Grade 3", "Grade 2", "Grade 2", "Grade 3", "Grade 3", "Grade 2", "Grade 3"],
    "Tumor_Volume_mm3": [120.5, 340.2, 115.0, 290.8, 150.3, 310.5, 118.2, 335.0],
    "Mean_Intensity": [85.4, 45.2, 88.1, 50.3, 80.5, 42.1, 86.0, 44.5]
}
df = pd.DataFrame(data)
# print(df)

# ---------------------------------------------------------
# Ví dụ 1: Gom nhóm theo 1 nhãn cột (MGMT_Status)
# Áp dụng hàm mean() cho các nhóm vừa tạo
# ---------------------------------------------------------
print("--- Phân tích trung bình theo Trạng thái MGMT ---")
df_mgmt_mean = df.groupby("MGMT_Status")[["Tumor_Volume_mm3", "Mean_Intensity"]].mean() # Chỉ lếy theo cột Tumor_Volume_mm3 và Mean_Intensity
print(df_mgmt_mean.reset_index())

# ---------------------------------------------------------
# Ví dụ 2: Gom nhóm theo nhiều cột (Tạo MultiIndex giống tài liệu)
# Áp dụng hàm sum() cho các nhóm đa biến
# ---------------------------------------------------------
print("\n--- Tổng đặc trưng theo Trạng thái MGMT và Cấp độ khối u ---")
df_multi_group = df.groupby(["MGMT_Status", "Tumor_Grade"])[["Tumor_Volume_mm3", "Mean_Intensity"]].sum()
print(df_multi_group.reset_index())

### Pandas Dataframe merge() Method
- pd.concat() (Nối dữ liệu): Cách hoạt động: Dán/Nối các bảng dữ liệu lại với nhau một cách cơ học (theo chiều dọc để thêm hàng, hoặc chiều ngang để thêm cột).
- pd.merge() (Kết hợp dữ liệu theo khóa): Cách hoạt động: Tìm kiếm và ghép nối các hàng từ hai bảng khác nhau dựa trên một cột chung (gọi là "khóa" - key). Nó hoạt động tương tự như VLOOKUP trong Excel hay JOIN trong SQL.

In [ ]:
import pandas as pd 

# ==========================================
# 1. CONCAT (NỐI DỮ LIỆU)
# ==========================================
print("--- 1. KẾT QUẢ SỬ DỤNG CONCAT ---")

batch01 = pd.DataFrame (
    {
        "Patient_ID": ["P01", "P02"],
        "MGMT_Status": ["Methylated", "Unmethylated"],
    }
)
batch02 = pd.DataFrame (
    {
        "Patient_ID": ["P03", "P04", "P05"],
        "MGMT_Status": ["Methylated", "Methylated", "Unmethylated"],
    }
)

# Dùng concat để xếp chồng đợt 2 xuống dưới đợt 1
clinical_data_full = pd.concat([batch01, batch02], ignore_index=True) # ignore_index=True giúp đánh lại số thứ tự (index) từ 0 cho liền mạch
print("Dữ liệu lâm sàng tổng hợp:\n", clinical_data_full)
print("\n")


# ==========================================
# 2. MERGE (GỘP DỮ LIỆU THEO KHÓA)
# ==========================================
print("--- 2. KẾT QUẢ SỬ DỤNG MERGE ---")

# Dữ liệu hình ảnh (Radiomics) được trích xuất từ MRI, chứa đặc trưng khối u
radiomics_data = pd.DataFrame({
    "Patient_ID": ["P01", "P02", "P03", "P04"],
    "Tumor_Volume_mm3": [120.5, 340.2, 115.0, 290.8],
    "Mean_Intensity": [85.4, 45.2, 88.1, 50.3]
})

# Dùng merge để gộp clinical_data_full và radiomics_data
# on="Patient_ID" chỉ định cột dùng làm khóa để so khớp
# how="inner" đảm bảo chỉ giữ lại những bệnh nhân có mặt ở cả 2 bảng - Giá trị mặc định 
final_dataset = pd.merge(clinical_data_full, radiomics_data, on="Patient_ID", how="inner")
print("Dữ liệu phân tích cuối cùng (Đã gộp):\n", final_dataset)

final_dataset = pd.merge(clinical_data_full, radiomics_data, on="Patient_ID", how="left") 
print("\nDữ liệu phân tích cuối cùng (Đã gộp - lấy bệnh nhân chỉ có tại clinical_data_full):\n", final_dataset)

# NOTE: Ngoài ra còn có các tùy chọn khác như how="right" (giữ tất cả bệnh nhân từ radiomics_data) hoặc how="outer" (giữ tất cả bệnh nhân từ cả 2 bảng, điền NaN nếu không có thông tin).

### Pandas Dataframe pivot_table() Method
pivot_table (Bảng tổng hợp) trong Pandas hoạt động tương tự như tính năng Pivot Table trong Excel. Nó giúp tái cấu trúc dữ liệu từ dạng danh sách dài thành dạng lưới 2 chiều (ma trận).

Khác biệt chính giữa groupby và pivot_table:
- groupby: Thường trả về dữ liệu dưới dạng cấu trúc đa tầng (MultiIndex) dọc theo các hàng. Rất tốt cho máy tính xử lý tiếp.
- pivot_table: Trải phẳng dữ liệu theo cả hàng (index) và cột (columns). Rất tốt để con người đọc, so sánh chéo và đưa vào các báo cáo nghiên cứu.

In [ ]:
import pandas as pd
import numpy as np

# 1. Khởi tạo dữ liệu
data = {
    "Patient_ID": ["P01", "P02", "P03", "P04", "P05", "P06"],
    "MGMT_Status": ["Methylated", "Unmethylated", "Methylated", "Unmethylated", "Methylated", "Unmethylated"],
    "Tumor_Grade": ["Grade 2", "Grade 3", "Grade 3", "Grade 2", "Grade 2", "Grade 3"],
    "Tumor_Volume_mm3": [120.5, 340.2, 150.3, 290.8, 115.0, 310.5],
    "Mean_Intensity": [85.4, 45.2, 80.5, 50.3, 88.1, 42.1]
}
df = pd.DataFrame(data)

# 2. Xây dựng Pivot Table
# Mục tiêu: Tạo ma trận phân tích Thể tích khối u
tumor_volume_pivot = pd.pivot_table(
    df,
    values="Tumor_Volume_mm3",  # Chỉ số dạng số cần phân tích
    index="MGMT_Status",  # Trục tung (Hàng): Các cấp độ khối u
    columns="Tumor_Grade",  # Trục hoành (Cột): Trạng thái gen
    aggfunc=np.mean  # Tính trung bình thể tích khối u cho mỗi nhóm - Giá trị mặc định 
)

print("--- Ma trận Thể tích khối u trung bình ---")
print(tumor_volume_pivot.reset_index())  # reset_index() để chuyển MGMT_Status từ index thành cột bình thường

# NOTE: Ngoài np.mean, có thể sử dụng các hàm tổng hợp khác như np.sum, np.median, np.max, np.min tùy theo mục đích phân tích.

### Pandas Dataframe apply() Method
Phương thức apply() trong Pandas cho phép áp dụng một hàm do tự viết (hoặc các hàm toán học có sẵn) lên một trục của DataFrame (hàng hoặc cột), hoặc lên một chuỗi (Series) dữ liệu.

Syntax cơ bản: dataframe.apply(func, axis, raw, result_type, args, kwds)

In [ ]:
import pandas as pd
import numpy as np

# 1. Khởi tạo dữ liệu
data = {
    "Patient_ID": ["P01", "P02", "P03", "P04", "P05", "P06"],
    "Tumor_Volume_mm3": [120.5, 340.2, 150.3, 290.8, 115.0, 310.5],
    "Mean_Intensity": [85.4, 45.2, 80.5, 50.3, 88.1, 42.1]
}
df = pd.DataFrame(data)

# ==========================================
# VÍ DỤ 1: SỬ DỤNG APPLY LÊN MỘT CỘT (SERIES)
# ==========================================
print("--- 1. Chuẩn hóa Thể tích khối u bằng Logarit ---")

# Lấy logarit tự nhiên (np.log) cho từng giá trị 'x' trong cột.
df["Log_Tumor_Volume"] = df["Tumor_Volume_mm3"].apply(lambda x : np.log(x)) 
# Hoặc có thể dùng: df["Log_Tumor_Volume"] = df["Tumor_Volume_mm3"].apply(np.log) vì np.log có sẵn hàm log

print(df[["Patient_ID", "Tumor_Volume_mm3", "Log_Tumor_Volume"]])
print("\n")


# ==========================================
# VÍ DỤ 2: SỬ DỤNG APPLY LÊN TỪNG HÀNG (DATAFRAME)
# ==========================================
print("--- 2. Phân loại Nguy cơ dựa trên nhiều biến ---")

# S1 : Định nghĩa hàm logic
def evaluate_rick(row):
    """
    Hàm đánh giá nguy cơ:
    Nếu thể tích > 200 && Cường độ < 60 -> High Risk
    """
    volume = row["Tumor_Volume_mm3"]
    intensity = row["Mean_Intensity"]
    
    if volume > 200 and intensity < 60:
        return "High Risk"
    else:
        return "Low Risk"
    
# S2 : Áp dụng hàm lên DataFrame với axis=1 
df["Risk_Category"] = df.apply(evaluate_rick, axis=1) # axis=1 chỉ định áp dụng theo hàng (row-wise)

print(df[["Patient_ID", "Tumor_Volume_mm3", "Mean_Intensity", "Risk_Category"]])

### Pandas Dataframe melt() Method
- Mục đích : Phương thức melt() trong Pandas được sử dụng để chuyển đổi một DataFrame từ định dạng "rộng" (wide format) sang định dạng "dài" (long format). Về cơ bản, có thể hiểu melt() chính là thao tác ngược lại hoàn toàn so với pivot_table().
- Ứng dụng thực tế : Các thư viện vẽ biểu đồ (như Seaborn) hoặc các mô hình Machine Learning thường yêu cầu dữ liệu đầu vào ở định dạng "dài".

In [ ]:
import pandas as pd

# ==========================================
# 1. KHỞI TẠO DỮ LIỆU GỐC (ĐỊNH DẠNG RỘNG)
# ==========================================
# Giả sử - Theo dõi kích thước khối u (Tumor_Volume) qua 3 giai đoạn
data = {
    "Patient_ID": ["P01", "P02", "P03"],
    "Month_1": [120.5, 340.2, 115.0],
    "Month_2": [130.0, 360.0, 110.0],
    "Month_3": [125.0, 350.0, 108.0]
}
df_wide = pd.DataFrame(data)

print("--- Dữ liệu gốc (Định dạng rộng) ---")
print(df_wide)
print("\n")

# ==========================================
# 2. SỬ DỤNG MELT() ĐỂ CHUYỂN SANG ĐỊNH DẠNG DÀI
# ==========================================

# df_long = df_wide.melt()
# print(df_long)

df_long = df_wide.melt(
    id_vars=["Patient_ID"],
    value_vars=["Month_1", "Month_2", "Month_3"],
    var_name="Time",
    value_name="Tumor_Volume"
)
print("--- Dữ liệu sau khi dùng melt() (Định dạng Dài) ---")
print(df_long)

### Pandas Dataframe value_counts() Method
value_counts() là một phương thức của Pandas dùng để đếm số lần xuất hiện của các giá trị duy nhất (unique values) trong một cột dữ liệu (Series). Nó giúp nhanh chóng nhìn thấy sự phân bổ của dữ liệu phân loại (categorical data).

In [ ]:
import pandas as pd

# KHỞI TẠO DỮ LIỆU
data = {
    "Patient_ID": ["P01", "P02", "P03", "P04", "P05", "P06", "P07"],
    "MGMT_Status": ["Methylated", "Unmethylated", "Unmethylated", "Methylated", "Unmethylated", "Unmethylated", None],
    "Tumor_Grade": ["Grade 2", "Grade 3", "Grade 3", "Grade 2", "Grade 3", "Grade 3", "Grade 2"]
}
df = pd.DataFrame(data)


print("--- 1. Đếm số lượng cơ bản (Mặc định) ---")
# Chọn cột MGMT_Status và gọi hàm value_counts()
dem_co_ban = df["MGMT_Status"].value_counts()
print(dem_co_ban)

print("\n--- 2. Tính tỷ lệ phần trăm (Tham số normalize) ---")
# normalize=True sẽ trả về tỷ lệ (từ 0 đến 1) thay vì số đếm tuyệt đối
ty_le = df["MGMT_Status"].value_counts(normalize=True)
print(ty_le * 100)  # Nhân với 100 để chuyển sang phần trăm

print("\n--- 3. Bao gồm giá trị NaN (Tham số dropna) ---")
# Mặc định Pandas sẽ bỏ qua các ô trống (None/NaN). 
# dropna=False ép Pandas phải đếm cả những bệnh nhân bị thiếu dữ liệu này.
dem_ca_non = df["MGMT_Status"].value_counts(dropna=False)
print(dem_ca_non)


### Pandas Dataframe fillna() Method
fillna() được sử dụng để tìm kiếm các giá trị bị rỗng (thường được biểu diễn là NaN hoặc None trong Pandas) và lấp đầy chúng bằng một giá trị cụ thể hoặc một phương pháp tính toán thống kê.

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. KHỞI TẠO DỮ LIỆU CÓ CHỨA GIÁ TRỊ RỖNG (NaN)
# ==========================================
data = {
    "Patient_ID": ["P01", "P02", "P03", "P04", "P05"],
    "MGMT_Status": ["Methylated", np.nan, "Unmethylated", "Methylated", np.nan],
    "Tumor_Volume_mm3": [120.5, 340.2, np.nan, 290.8, 115.0]
}
df = pd.DataFrame(data)

print("--- Dữ liệu gốc chứa NaN ---")
print(df)
print("\n")

# ==========================================
# 2. XỬ LÝ DỮ LIỆU BẰNG FILLNA()
# ==========================================
# C1 : Điền giá trị phân loại bị thiếu bằng một chuỗi ký tự cố định
df["MGMT_Status_Filled"] = df["MGMT_Status"].fillna("Unknown")

# C2: Điền giá trị số học bị thiếu bằng trung vị (Median)
# Sử dụng median thường an toàn hơn mean trong dữ liệu y tế vì nó ít bị ảnh hưởng bởi các giá trị ngoại lai (outliers)
median_volume = df["Tumor_Volume_mm3"].median()
df["Tumor_Volume_Filled"] = df["Tumor_Volume_mm3"].fillna(median_volume)

print("--- Dữ liệu sau khi lấp đầy từng cột ---")
print(df[["Patient_ID", "MGMT_Status_Filled", "Tumor_Volume_Filled"]])
print("\n")

# ==========================================
# 3. KỸ THUẬT NÂNG CAO: ĐIỀN NHIỀU CỘT CÙNG LÚC
# ==========================================
# Sử dụng Dictionary (Từ điển) để áp dụng các luật điền dữ liệu khác nhau cho từng cột
fill_rules = {
    "MGMT_Status" : "Unknown",
    "Tumor_Volume_mm3" : df["Tumor_Volume_mm3"].mean()
}
# Tham số inplace=True sẽ ghi đè trực tiếp lên DataFrame gốc thay vì tạo ra một bản sao mới
df.fillna(value=fill_rules, inplace=True)

print("--- Dữ liệu gốc sau khi dùng inplace=True ---")
print(df[["Patient_ID", "MGMT_Status", "Tumor_Volume_mm3"]])

### Pandas Dataframe sort_values() Method
sort_values() dùng để sắp xếp các hàng của một DataFrame dựa trên giá trị của một hoặc nhiều cột. -> Cách hoạt động giống với tính năng "Sort A to Z" hoặc "Sort Largest to Smallest" trong Excel.

In [ ]:
import pandas as pd

# ==========================================
# 1. KHỞI TẠO DỮ LIỆU
# ==========================================
data = {
    "Patient_ID" : ["P01", "P02", "P03", "P04", "P05", "P06"],
    "MGMT_Status" : ["Methylated", "Unmethylated", "Methylated", "Unmethylated", "Methylated", "Unmethylated"],
    "Tumor_Grade": ["Grade 2", "Grade 3", "Grade 3", "Grade 2", "Grade 3", "Grade 2"],
    "Tumor_Volume_mm3": [120.5, 340.2, 310.5, 290.8, 150.3, 115.0],
    "Mean_Intensity": [85.4, 45.2, 42.1, 50.3, 80.5, 88.1]
}
df = pd.DataFrame(data)

print("--- 1. Dữ liệu gốc ---")
print(df[["Patient_ID", "Tumor_Grade", "Tumor_Volume_mm3", "Mean_Intensity"]])

# ==========================================
# 2. SẮP XẾP THEO 1 CỘT (ĐƠN BIẾN)
# ==========================================
print("\n--- 2. Sắp xếp thể tích khối u từ Lớn đến Nhỏ ---")
# Sử dụng ascending=False để sắp xếp giảm dần 
df_sorted_volume = df.sort_values(by="Tumor_Volume_mm3", ascending=False)
print(df_sorted_volume[["Patient_ID", "Tumor_Volume_mm3"]])

# ==========================================
# 3. SẮP XẾP THEO NHIỀU CỘT (ĐA BIẾN)
# ==========================================
print("\n--- 3. Sắp xếp theo Tumor_Grade (Tăng dần), sau đó theo Mean_Intensity (Giảm dần) ---")
df_mutil_sort = df.sort_values(
    by=["Tumor_Grade", "Mean_Intensity"],
    ascending=[True, False ]
)

print(df_mutil_sort[["Patient_ID", "Tumor_Grade", "Mean_Intensity"]])


### Numpy np.dot() Method
np.dot() là hàm cốt lõi của numpy dùng để tính tích vô hướng (Dot product) của 2 vector (mảng 1 chiều), hoặc nhân ma trận (Matrix Multiplication) của các mảng đa chiều.

In [20]:
import numpy as np
import pandas as pd

# ==========================================
# 1. TÍNH TÍCH VÔ HƯỚNG 1D (MỘT BỆNH NHÂN)
# ==========================================
print("--- 1. Tính điểm dự đoán cho 1 bệnh nhân ---")

# Giả sử Bệnh nhân P01 có 2 đặc trưng: Thể tích khối u & Cường độ tín hiệu
patient_features = np.array([120.5, 85.4])

# Mô hình AI gán trọng số: Thể tích chiếm 0.4, Cường độ chiếm 0.6
weights = np.array([0.4, 0.6])

# Sử dụng np.dot() để tính tổng điểm 
# Phép tính : 120.5*0.4 + 85.4*0.6
risk_score = np.dot(patient_features, weights)

print(f"Đặc trưng bệnh nhân: {patient_features}")
print(f"Trọng số mô hình: {weights}")
print(f"Điểm nguy cơ (Risk Score): {risk_score}\n")

# ==========================================
# 2. NHÂN MA TRẬN 2D (NHIỀU BỆNH NHÂN CÙNG LÚC)
# ==========================================
print("--- 2. Tính điểm dự đoán cho hàng loạt bệnh nhân ---")

# Ma trận 3x2: 3 Bệnh nhân (hàng), 2 Đặc trưng (cộ
patients_matrix = np.array([
    [120.5, 85.4],
    [340.2, 45.2],
    [150.3, 80.5]
])

# Khi nhân ma trận (3x2) với vector trọng số (2,) -> np.dot() sẽ tự động tính điểm cho cả 3 bệnh nhân
all_scores = np.dot(patients_matrix, weights)

result_df = pd.DataFrame(
    {
        "Patient_ID": ["P01", "P02", "P03"],
        "Risk_Scores": all_scores
    }
)

print("Ma trận dữ liệu bệnh nhân:\n", patients_matrix)
print("Danh sách điểm số dự đoán cho P01, P02, P03 tương ứng:")
print(result_df) 

--- 1. Tính điểm dự đoán cho 1 bệnh nhân ---
Đặc trưng bệnh nhân: [120.5  85.4]
Trọng số mô hình: [0.4 0.6]
Điểm nguy cơ (Risk Score): 99.44

--- 2. Tính điểm dự đoán cho hàng loạt bệnh nhân ---
Ma trận dữ liệu bệnh nhân:
 [[120.5  85.4]
 [340.2  45.2]
 [150.3  80.5]]
Danh sách điểm số dự đoán cho P01, P02, P03 tương ứng:
  Patient_ID  Risk_Scores
0        P01        99.44
1        P02       163.20
2        P03       108.42
